# Ramen V5：语义修复与独立基线对照
选择 **L4 GPU**（运行时 → 更改运行时类型）。本 notebook 不选择 A100、不购买额度、不删除旧结果。

顺序：环境检查 → 新目录复制数据 → 8 候选图 teacher → 真实 GPU 300 步链路测试 → 全量 teacher → 全新 15000 步上限实验与基线 → 查看结果 → 可选人工备份。

V5：RGB 预热 2500 步，语义权重用 2000 步渐增；32D 语言与 16D 粗/中/细 affinity 分开，默认语义不改变几何，RGB 继续优化几何。参考 LaGa/SAGA 的独立层级特征思想，不等于完整复现论文。300 步使用已有 RGB 几何，仅检查 CUDA、梯度和导出，**不能据此宣称质量达到论文水平**。

所有新数据与训练输出位于 `/content/ramen_semantic_v5`。Colab 重置会丢失 /content；本版不在后台反复轮换云盘 checkpoint。云盘当前约 5.2 GiB 的限制不阻止本地 smoke；正式训练前仅检查本地空间，最后备份必须按实际文件逐项估算。

8 候选图中 2 张作验证、6 张作训练；保留全量重建得到的 COLMAP 几何和成熟 RGB，因此不是独立稀疏视角实验。正式实验重新使用全量候选图和 12 张验证图，绝不沿用子集 teacher。


In [ ]:
import importlib.util, json, os, subprocess, sys, shutil, hashlib, urllib.request, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), '请先连接 GPU 运行时'
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu, '| PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
assert 'L4' in gpu, '此 notebook 以 L4 为目标；请手动选 L4 后继续，避免误用高价卡'
REPO = Path('/content/gaussian-splatting')
if (REPO / '.git').is_dir():
    dirty = subprocess.check_output(['git','status','--porcelain'], cwd=REPO, text=True)
    assert not dirty.strip(), '仓库有本地修改：先保留/处理，不自动覆盖'
    subprocess.run(['git','pull','--ff-only'], cwd=REPO, check=True)
else:
    subprocess.run(['git','clone','--recursive','https://github.com/Xuyw041006-arch/gaussian-splatting.git',str(REPO)], check=True)
subprocess.run(['git','submodule','update','--init','--recursive'], cwd=REPO, check=True)
packages = {'plyfile':'plyfile','open_clip':'open-clip-torch','sklearn':'scikit-learn','ftfy':'ftfy','cv2':'opencv-python-headless','segment_anything':'git+https://github.com/facebookresearch/segment-anything.git','lpips':'lpips','tqdm':'tqdm'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing], check=True)
FORCE_REBUILD_EXTENSIONS = False  # 若更换了 PyTorch/CUDA 且遇 ABI 错误，再改 True
extensions = {'diff_gaussian_rasterization':'diff-gaussian-rasterization','simple_knn':'simple-knn','fused_ssim':'fused-ssim'}
for module, folder in extensions.items():
    if FORCE_REBUILD_EXTENSIONS or importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation',str(REPO/'submodules'/folder)], check=True)
sys.path.insert(0, str(REPO))
print('Git:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())
def run(command):
    print('运行:', ' '.join(str(value) for value in command), flush=True)
    subprocess.run([str(value) for value in command], cwd=REPO, check=True)
run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_semantic_v5.py','-v'])
run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_preprocess_teacher.py','-v'])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from scripts.run_ramen_recovery import copy_scene, atomic_copy
PERSIST = Path('/content/drive/MyDrive/semantic_adaptive_3dgs')
WORK = Path('/content/ramen_semantic_v5')
SCENE = WORK / 'data/ramen'
OUTPUT = WORK / 'outputs_full'
SOURCE_SCENE = PERSIST / 'ramen_detail_v2_15k/data/ramen'
OLD_RGB_PLY = PERSIST / 'ramen_curriculum_v3_15k/outputs_full/sequential/point_cloud/iteration_7000/point_cloud.ply'
assert SOURCE_SCENE.is_dir(), f'缺少旧数据: {SOURCE_SCENE}'
WORK.mkdir(parents=True, exist_ok=True)
print('本地可用 GiB:', round(shutil.disk_usage(WORK).free/2**30, 2))
print('Drive 挂载层可用 GiB:', round(shutil.disk_usage(PERSIST).free/2**30, 2))
print('注意：DriveFS 数值不一定等于账户真实配额；回收站仍可能占用配额。')
required = sum(p.stat().st_size for name in ('images','images_train','sparse','test_mask') for p in (SOURCE_SCENE/name).rglob('*') if p.is_file())
assert shutil.disk_usage(WORK).free > required + 3*2**30, '本地空间不足以安全复制数据和 SAM'
if not (SCENE/'semantic_meta.npz').is_file():
    copy_scene(SOURCE_SCENE, SCENE)  # 仅 images/images_train/sparse/test_mask；不复制旧语义
else:
    assert json.loads((SCENE/'recovery_scene_provenance.json').read_text())['source']==str(SOURCE_SCENE.resolve()), '已有冻结场景来源不符'
    print('保留已有冻结场景；不重新覆盖 train/val/test 清单。')
assert OLD_RGB_PLY.is_file(), f'300 步诊断需要这个旧 RGB 权重，请检查路径: {OLD_RGB_PLY}'
# 在加载完整模型前确认是原始 SH3 格式（45 个 f_rest 字段）。
properties = []
with OLD_RGB_PLY.open('rb') as handle:
    for _ in range(1000):
        line = handle.readline().decode('ascii').strip()
        if line == 'end_header': break
        if line.startswith('property ') and line.split()[-1].startswith('f_rest_'):
            properties.append(line)
assert len(properties) == 45, '旧 PLY 不是 SH3，不能直接拿 SH5 历史权重运行此诊断'
RGB_PLY = WORK / 'inputs/mature_rgb_7000.ply'
if not RGB_PLY.is_file():
    atomic_copy(OLD_RGB_PLY, RGB_PLY)
print('新场景:', SCENE, '| 旧权重只读复制:', RGB_PLY)


In [ ]:
SAM_SHA256 = 'a7bf3b02f3ebf1267aba913ff637d9a2d5c33d3173bb679e46d9f338c26f262e'
candidates = [
    Path('/content/ramen_recoverable_cache/ramen_detail_v2_15k/assets/sam_vit_h_4b8939.pth'),
    Path('/content/ramen_assets/sam_vit_h_4b8939.pth'),
    WORK/'assets/sam_vit_h_4b8939.pth',
]
SAM = next((path for path in candidates if path.is_file()), candidates[-1])
if not SAM.is_file():
    SAM.parent.mkdir(parents=True, exist_ok=True)
    partial = SAM.with_suffix('.download')
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth', partial)
    partial.replace(SAM)
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()
assert sha256(SAM) == SAM_SHA256, 'SAM 校验失败；不开始训练，也不自动删除缓存'
print('SAM SHA256 校验完成:', SAM)
BENCH = [sys.executable,'-u','scripts/run_ramen_benchmark.py',
         '--scene',SCENE,'--sam_checkpoint',SAM,'--output_root',OUTPUT,
         '--semantic_protocol','v5','--iterations','15000','--semantic_iterations','5000',
         '--semantic_start','2500','--semantic_ramp_iterations','2000',
         '--feature_dim','32','--feature_width','512','--validation_views','12',
         '--validation_interval','1000','--early_stop_patience','4','--sam_crop_n_layers','1']


## 先准备低成本诊断 teacher
先只对 8 张候选图生成 teacher（6 训练 + 2 验证），SAM crop 层数为 0。正式全量 teacher 仍使用 crop 层数 1，并且在 smoke 通过后才生成；不能把子集 PCA/teacher 接到正式训练。

SAM crop 层数 1 除全图外还处理裁剪区域，当前每个裁剪维持相同采样网格，明显更慢。max_masks=192 是输出筛选上限，不会减少 SAM 候选生成成本。CLIP 方形 crop/背景视图编码也增加前处理时间。两模型共享同一冻结 teacher，teacher 时间单独记录，不计入目前“等训练时间”指标。


In [ ]:
from scripts.prepare_ramen_smoke_subset import prepare as prepare_smoke_subset
SMOKE_SCENE = WORK/'smoke_data/ramen_8candidate'
SMOKE_TEACHER_OUTPUT = WORK/'smoke_teacher_8candidate'
if not SMOKE_SCENE.exists():
    prepare_smoke_subset(SCENE, SMOKE_SCENE, views=8)
else:
    provenance = json.loads((SMOKE_SCENE/'smoke_subset_provenance.json').read_text())
    assert provenance['source'] == str(SCENE.resolve()) and len(provenance['non_test_candidate_views']) == 8, '已有子集身份不符，不覆盖'
def replace_flag(command, flag, value):
    result = list(command)
    result[result.index(flag)+1] = value
    return result
SMOKE_BENCH = replace_flag(replace_flag(replace_flag(replace_flag(BENCH,
    '--scene',SMOKE_SCENE),'--output_root',SMOKE_TEACHER_OUTPUT),
    '--validation_views','2'),'--sam_crop_n_layers','0')
started = time.monotonic()
run([*SMOKE_BENCH,'--prepare_only','--resume'])
print('子集准备用时（含复用校验）秒:', round(time.monotonic()-started,2))
for name in ('train','val','test'):
    print('smoke', name, len((SMOKE_SCENE/f'sparse/0/{name}.txt').read_text().splitlines()))
assert (SMOKE_TEACHER_OUTPUT/'v5_teacher_files.json').is_file()
print('这是全量几何辅助的 CUDA 诊断，不是 8 视角重建成绩。')


## 300 步真实 GPU 链路诊断
加载旧的成熟 RGB 几何，语义重新初始化；语义从第 0 步开始，渐增 100 步，关闭分裂和透明度重置。这样可以检查新分支的 CUDA 反传，不用把几何从头练起来。它与下面正式实验的预热设置不同，不纳入等时或质量对比。

本单元仅使用独立 8 候选图子集，输出也独立。teacher、RGB 来源或代码版本变化时拒绝复用现有 smoke 权重；请改新目录名，不清空旧结果。


In [ ]:
from scripts.run_ramen_recovery import complete_torch_archive
SMOKE = WORK/'gpu_smoke_8candidate'
SMOKE.mkdir(parents=True, exist_ok=True)
smoke_command = [sys.executable,'-u','train.py','-s',SMOKE_SCENE,'-m',SMOKE,
    '--eval','--validation_file',SMOKE_SCENE/'sparse/0/val.txt',
    '--sh_degree','3','--joint_semantics','--semantic_protocol','v5',
    '--semantic_start','0','--semantic_ramp_iterations','100',
    '--semantic_weight','.12','--rgb_tier_weights','.75','1','2',
    '--semantic_tier_weights','.5','1','2','--semantic_boundary_weight','.06',
    '--importance_mask_dir',SMOKE_SCENE/'importance_masks',
    '--iterations','300','--save_iterations','300','--test_iterations','300',
    '--checkpoint_iterations','300','--densify_until_iter','0',
    '--opacity_reset_interval','100000','--disable_viewer']
smoke_protocol = {'scope':'CUDA_only_not_sparse_benchmark','iterations':300,
    'rgb_sha256':sha256(RGB_PLY),'teacher_sha256':sha256(SMOKE_TEACHER_OUTPUT/'v5_teacher_files.json'),
    'pca_sha256':sha256(SMOKE_SCENE/'semantic_meta.npz'),
    'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()}
protocol_path = SMOKE/'smoke_protocol.json'
has_weights = any(SMOKE.glob('chkpnt*.pth')) or any(SMOKE.glob('semantic/iteration_*/*.pt'))
if has_weights:
    assert protocol_path.is_file() and json.loads(protocol_path.read_text()) == smoke_protocol, 'smoke 代码/teacher/RGB 来源改变；用新目录，不能混用权重'
else:
    protocol_path.write_text(json.dumps(smoke_protocol,indent=2))
completed = SMOKE/'semantic/iteration_300/semantic_features.pt'
checkpoints = sorted((p for p in SMOKE.glob('chkpnt[0-9]*.pth') if complete_torch_archive(p)), key=lambda p:int(p.stem[6:]))
if completed.is_file() and complete_torch_archive(completed):
    print('此目录已有 300 步诊断结果；将检查文件，不冒充新训练。')
else:
    if checkpoints:
        smoke_command += ['--start_checkpoint', checkpoints[-1]]
    else:
        smoke_command += ['--init_rgb_ply', RGB_PLY]
    run(smoke_command)
artifact = torch.load(completed, map_location='cpu', weights_only=False)
assert artifact.get('semantic_protocol') == 'v5'
assert artifact['features'].shape[1] == 32 and artifact['affinity_features'].shape[1] == 16
assert torch.isfinite(artifact['features']).all() and torch.isfinite(artifact['affinity_features']).all()
assert tuple(artifact['affinity_prefix_dimensions']) == (8,12,16)
assert 'scale_gate' not in artifact
assert (SMOKE/'point_cloud/iteration_300/point_cloud.ply').is_file(), '缺少匹配的 RGB PLY'
print('真实 GPU smoke 导出检查通过；这不是质量达标结论。')
print('语言/affinity:', tuple(artifact['features'].shape), tuple(artifact['affinity_features'].shape))
del artifact
torch.cuda.empty_cache()


In [ ]:
has_full_state = any(OUTPUT.glob('*/chkpnt*.pth')) or any(OUTPUT.glob('*/point_cloud/iteration_*/*.ply'))
assert (SMOKE/'semantic/iteration_300/semantic_features.pt').is_file() or has_full_state, '新实验先通过独立 smoke；恢复完整实验无需重做 smoke'
assert SCENE != SMOKE_SCENE and OUTPUT != SMOKE_TEACHER_OUTPUT
assert not (SCENE/'smoke_subset_provenance.json').exists(), '禁止把诊断子集当正式场景'
assert len(list((SCENE/'images_train').glob('*.*'))) > 12, '正式场景缺少全量训练候选图'
started = time.monotonic()
run([*BENCH,'--prepare_only','--resume'])
teacher_seconds = time.monotonic()-started
print('正式 teacher 准备/复用校验用时秒:', round(teacher_seconds,2))
for name in ('train','val','test'):
    print('full',name,len((SCENE/f'sparse/0/{name}.txt').read_text().splitlines()))
assert len((SCENE/'sparse/0/val.txt').read_text().splitlines()) == 12
assert (OUTPUT/'v5_teacher_files.json').is_file()
for name in ('semantic_summary.json','scene_inventory.json'):
    path=SCENE/name
    if path.is_file(): print(name, path.read_text()[:16000])
print('提示词是候选而非真值；正式训练前核查重要物体、普通物体、背景。改变 teacher 必须开新实验。')


## 正式 V5 与基线
下面从场景初始点云独立训练，不读取 smoke 的成熟 RGB 权重。上限 15000 步，RGB 预热 2500 步、渐增 2000 步；12 个验证视角用于早停/最佳权重选择。联合模型与纯 RGB→顺序语义基线使用同一数据、同一 GPU，等时预算是否真正满足以最终计时协议为准，不能只看请求了“等时”。

运行可能需要数小时并消耗 GPU 额度；本地磁盘和会话中断风险仍存在。云盘不足不阻止本地训练，但未成功人工备份的结果会随运行时重置丢失。

公平性边界：基线是原始 3DGS RGB 加本仓库顺序语义分支，并非官方 LangSplat/LaGa/SAGA 复现。V5 有额外 affinity，预算比较衡量整体方案效率而不是单一机制消融。基线 RGB 训练完后只把联合模型剩余时间分给语义，最多 15000 个语义优化步；若剩余预算不足、缺少历史计时或达到上限，不能保证等时。相同评测协议下才比较；前处理、下载、最终测试耗时未纳入当前训练预算。断线可继续不代表仍可认证等时。


In [ ]:
current_bytes = sum(p.stat().st_size for p in WORK.rglob('*') if p.is_file())
local_free = shutil.disk_usage(WORK).free
# 按已知成熟 PLY 体积保守估计两个模型的优化器、best/中间 checkpoint、最终文件。
# 高斯数量可能增长，因此这是提示，不是存储上界，更不是 Drive 配额要求。
estimate = max(12*2**30, RGB_PLY.stat().st_size * 36)
print('当前本地用量 GiB:', round(current_bytes/2**30,2))
print('估计新增本地空间 GiB:', round(estimate/2**30,2), '| 可用:', round(local_free/2**30,2))
ALLOW_LOW_LOCAL_SPACE = False
assert local_free >= estimate or ALLOW_LOW_LOCAL_SPACE, '本地空间低于保守估计。检查后才可手动放行；不自动删除任何文件'
print('本次正式训练只写 /content，不要求 Drive 额外 8 GiB；未备份有会话丢失风险。')
run([*BENCH,'--resume','--skip_preprocess'])
comparison_path = OUTPUT/'comparison.json'
assert comparison_path.is_file(), '最终 comparison.json 尚未生成，不能宣称全部完成'
print('正式实验与评估流程返回成功；下一单元查看完整协议和指标。')


In [ ]:
from IPython.display import Markdown, display
comparison_path = OUTPUT/'comparison.json'
assert comparison_path.is_file(), '结果尚未生成'
comparison = json.loads(comparison_path.read_text())
print(json.dumps(comparison, ensure_ascii=False, indent=2))
run([sys.executable,'scripts/build_ramen_report.py','--output_root',OUTPUT,'--report_dir',OUTPUT/'report'])
report = OUTPUT/'report/ramen_final_report.md'
if report.is_file():
    display(Markdown(report.read_text()))
print('注意：不同测试视角/阈值/评分协议的历史数值不能直接相减；未满足等时容差不能宣称严格等时。')


## 可选的多视角描述符库消融（默认关闭）
使用训练视角中已对齐的 SAM 区域描述符构建独立 bank，再单独评估。它是 LaGa 启发的 region-to-affinity 检索扩展，不是完整逐物体分解/自适应聚类复现，也不替代主 comparison.json。

该步骤增加训练后计算量，必须单列耗时；不能把这份结果当原先等时预算内的性能。阈值预先固定为 0.25，不根据测试集调阈值；主模型仍用原有 clip_relevancy 协议。


In [ ]:
RUN_DESCRIPTOR_ABLATION = False
if RUN_DESCRIPTOR_ABLATION:
    assert (OUTPUT/'comparison.json').is_file(), '先完成主比较'
    BANK=OUTPUT/'descriptor_bank_joint_15000.npz'
    EXTRA_EVAL=OUTPUT/'eval_joint_descriptor_bank'
    started=time.monotonic()
    if not BANK.exists():
        run([sys.executable,'scripts/build_semantic_descriptor_bank.py','--model',OUTPUT/'joint',
             '--iteration','15000','--output',BANK,'--max_views','24',
             '--max_regions_per_view','96','--max_records','8192'])
    run([sys.executable,'-m','scripts.evaluate_lerf_mask','--model',OUTPUT/'joint',
         '--iteration','15000','--test_mask',SCENE/'test_mask','--descriptor_bank',BANK,
         '--mask_protocol','gg_native','--threshold','.25','--granularity','1',
         '--output',EXTRA_EVAL,'--important_labels','egg,pork belly,wavy noodles in bowl',
         '--normal_labels','yellow bowl,chopsticks,glass of water'])
    extra={'additional_postprocessing_and_evaluation_seconds':time.monotonic()-started,
           'outside_primary_training_budget':True,'main_comparison_unchanged':True}
    (OUTPUT/'descriptor_bank_ablation_timing.json').write_text(json.dumps(extra,indent=2))
    print((EXTRA_EVAL/'metrics.json').read_text())
    print(json.dumps(extra,indent=2))
else:
    print('未运行描述符库消融；主 comparison.json 不变。')


## 可选人工备份：默认关闭，不轮换、不删旧模型
先保持 RUN_SELECTED_BACKUP=False 查看实际选中文件和字节数；确认 Google 账户真实配额足够再开启。BACKUP_MODE='final' 备份最终模型/语义/评估；中断后可选 'resume'，无需 comparison.json，备份最近完整 checkpoint（及 best）、顺序语义优化器状态、完成标记、冻结 teacher 与必要 maps。

训练进程仍在写 checkpoint 时不要复制；先停止当前单元或等待其正常结束，再运行备份单元。它不自动定时备份，未成功备份前，运行时销毁仍会丢失全部 /content 结果。约 5.2 GiB 可能不足，必须按实际体积决定，不删除旧模型或回收站。

数据 images/sparse/GT 仍引用原 Drive 数据；SAM/CLIP 权重不重复备份。恢复到相同 /content 路径，先复制原数据，再运行下方可选恢复单元，核验冻结 maps 后才 --resume；不要重拟合 PCA。DriveFS 回读成功仍需要在 Drive 页面确认同步，不能据此保证服务端已落盘。


In [ ]:
from datetime import datetime, timezone
from scripts.run_ramen_recovery import complete_torch_archive
RUN_SELECTED_BACKUP = False
BACKUP_MODE = 'final'  # final：完整结果；resume：中断后最近完整状态
def select_backup_files(work, scene, output, mode='final'):
    work, scene, output = Path(work), Path(scene), Path(output)
    if mode not in ('final','resume'): raise ValueError('Unknown backup mode')
    if mode == 'final' and not (output/'comparison.json').is_file():
        raise RuntimeError('最终对比尚未完成；中断后请显式选 resume 模式')
    selected, resume_aliases = {}, []
    def add(path, required=False):
        path=Path(path)
        if not path.is_file():
            if required: raise FileNotFoundError(path)
            return
        if path.is_symlink(): raise ValueError(f'拒绝符号链接: {path}')
        selected[str(path.relative_to(work))]=path
    def tree(path, required=False):
        if required and not Path(path).is_dir(): raise FileNotFoundError(path)
        for item in sorted(Path(path).rglob('*')):
            if item.is_file(): add(item)
    model_count=0
    for name in ('joint','sequential'):
        model=output/name
        numeric=sorted((p for p in model.glob('chkpnt[0-9]*.pth') if complete_torch_archive(p)),key=lambda p:int(p.stem[6:]))
        best=model/'best_val_chkpnt.pth'
        best=best if best.is_file() and complete_torch_archive(best) else None
        plys=sorted(model.glob('point_cloud/iteration_*/point_cloud.ply'),key=lambda p:int(p.parent.name.split('_')[-1]))
        if not numeric and best is None and not plys:
            if mode=='final': raise RuntimeError(f'{name} 缺少训练结果')
            continue
        model_count+=1
        if numeric: add(numeric[-1])
        if best is not None: add(best)
        if mode=='resume' and not numeric and best is not None:
            summary=json.loads((model/'validation_summary.json').read_text())
            step=int(summary['best_iteration'])
            if not 0 < step < 15000: raise RuntimeError('无法为 best checkpoint 建立安全续训别名')
            resume_aliases.append({'source':str(best.relative_to(work)),'target':str((model/f'chkpnt{step}.pth').relative_to(work))})
        if plys:
            final_ply=plys[-1]; add(final_ply)
            semantic=model/'semantic'/final_ply.parent.name
            add(semantic/'semantic_features.pt',required=mode=='final')
        # Baseline feature Adam/elapsed-loop state and completion flags are
        # essential: exporting only semantic_features.pt cannot resume it.
        for path in model.glob('semantic/iteration_*/semantic_checkpoint.pt'):
            if complete_torch_archive(path): add(path)
        for path in model.glob('semantic/iteration_*/*.json'): add(path)
        for path in model.glob('*.json'): add(path)
        for path in model.glob('*.jsonl'): add(path)  # V5 losses/gradient diagnostics
        for path in model.glob('semantic/iteration_*/descriptor_bank*.pt'): add(path)
        add(model/'cfg_args',required=True)
    if not model_count: raise RuntimeError('没有可备份的训练状态')
    for path in output.glob('*.json'): add(path)
    for path in output.glob('descriptor_bank*.npz'): add(path)
    for name in ('report','eval_joint','eval_sequential','eval_joint_descriptor_bank','teacher_reference'): tree(output/name)
    for name in ('semantic_meta.npz','semantic_summary.json','scene_inventory.json','sparse/0/train.txt','sparse/0/val.txt','sparse/0/test.txt'):
        add(scene/name,required=True)
    add(scene/'recovery_scene_provenance.json')
    for name in ('semantic_maps','importance_masks','semantic_raw'): tree(scene/name,required=True)
    for name in ('detail_weights','boundary_masks'): tree(scene/name)
    add(output/'v5_teacher_files.json',required=True)
    add(output/'experiment_protocol.json',required=True)
    return selected,resume_aliases
selected,resume_aliases=select_backup_files(WORK,SCENE,OUTPUT,BACKUP_MODE)
total=sum(path.stat().st_size for path in selected.values())
reserve=max(256*2**20,int(total*.05))
free=shutil.disk_usage(PERSIST).free
print('选中文件:',len(selected),'| 精确有效载荷:',total,'bytes =',round(total/2**30,3),'GiB')
print('含余量:',round((total+reserve)/2**30,3),'GiB | 挂载层可用:',round(free/2**30,3),'GiB')
print('还须核对 Google 账户真实配额。模式:',BACKUP_MODE)
if not RUN_SELECTED_BACKUP:
    print('仅预览，没有上传/删除结果。')
else:
    assert free>=total+reserve,'空间不足：未开始复制，本地结果保留'
    destination=PERSIST/('ramen_semantic_v5_selected_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    assert not destination.exists(),'拒绝覆盖已有备份'
    status={'backed_up':False,'backup_mode':BACKUP_MODE,'destination':str(destination),
            'selected_bytes':total,'source_dataset_on_drive':str(SOURCE_SCENE),
            'local_reset_risk':True,'server_side_sync_verified':False,'files':{},
            'resume_aliases':resume_aliases,'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()}
    status_file=WORK/'selected_backup_status.json'
    status_file.write_text(json.dumps(status,ensure_ascii=False,indent=2))
    try:
        destination.mkdir(parents=True)
        for relative,source in sorted(selected.items()):
            target=destination/relative
            metadata=atomic_copy(source,target,verify_checkpoint=source.suffix in ('.pth','.pt'))
            assert sha256(target)==metadata['sha256'],f'复制后校验失败: {target}'
            status['files'][relative]=metadata
        status.update(backed_up=True,note='挂载写入和回读校验完成；云端同步仍须人工确认')
        status_file.write_text(json.dumps(status,ensure_ascii=False,indent=2))
        atomic_copy(status_file,destination/'selected_backup_status.json')
        print('选中备份已写入并回读验证:',destination,'；确认云端同步后再断开。')
    except Exception as error:
        status.update(backed_up=False,error=repr(error))
        status_file.write_text(json.dumps(status,ensure_ascii=False,indent=2))
        print('备份失败或不完整！本地未删；部分副本可能占配额。',repr(error))
        raise


## 可选：运行时重置后恢复
先运行环境、原数据复制、SAM 单元，再运行此单元；不要先重新生成正式 teacher。只接受完整备份清单，逐文件 SHA256 验证，现有同名文件内容不一致则停止而不覆盖。恢复后运行“正式 teacher”单元的 --prepare_only --resume 校验，随后正式训练 --resume。

若只是浏览器断连而 /content 仍存在，不需要此恢复步骤，重新连接后检查进程，避免同时启动两份训练。恢复后的缺失计时历史会被标为不可认证等时。恢复数据不自动回退代码版本：必须与备份 commit 一致。


In [ ]:
RESTORE_FROM = ''  # 填入本 notebook 创建的 Drive 备份目录；留空不做任何操作
if RESTORE_FROM:
    backup=Path(RESTORE_FROM).resolve()
    assert backup.parent==PERSIST.resolve() and backup.name.startswith('ramen_semantic_v5_selected_'),'只恢复明确选中的本实验备份'
    manifest=json.loads((backup/'selected_backup_status.json').read_text())
    assert manifest.get('backed_up') is True,'备份未完成，不能作为完整恢复源'
    assert manifest['source_dataset_on_drive']==str(SOURCE_SCENE),'源数据身份不匹配'
    current_commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
    assert manifest.get('git_commit')==current_commit,'代码版本不同；先核查并选择匹配版本，不自动覆盖仓库'
    def safe_relative(relative):
        path=Path(relative)
        assert not path.is_absolute() and '..' not in path.parts and str(path) not in ('','.'),'非法备份路径'
        return path
    pending=[]
    for relative,metadata in manifest['files'].items():
        relative=safe_relative(relative); source=backup/relative; target=WORK/relative
        assert source.is_file() and not source.is_symlink() and sha256(source)==metadata['sha256'],f'备份损坏: {source}'
        if target.exists():
            assert target.is_file() and not target.is_symlink(),f'非法本地目标: {target}'
            if sha256(target)!=metadata['sha256']:
                original_split=SOURCE_SCENE/'sparse/0'/target.name
                is_copied_split=relative.parent==Path('data/ramen/sparse/0') and target.name in ('train.txt','val.txt','test.txt')
                assert is_copied_split and original_split.is_file() and sha256(target)==sha256(original_split),f'本地已有不同内容，拒绝覆盖: {target}'
                pending.append((source,target))  # 仅替换已验证为原数据副本的旧 split
        else: pending.append((source,target))
    required=sum(source.stat().st_size for source,_ in pending)
    required+=sum((backup/safe_relative(alias['source'])).stat().st_size for alias in manifest.get('resume_aliases',[]))
    assert shutil.disk_usage(WORK).free>required+256*2**20,'本地恢复空间不足'
    for source,target in pending:
        atomic_copy(source,target,verify_checkpoint=source.suffix in ('.pth','.pt'))
        assert sha256(target)==sha256(source),'恢复后校验失败'
    for alias in manifest.get('resume_aliases',[]):
        source=WORK/safe_relative(alias['source']); target=WORK/safe_relative(alias['target'])
        if target.exists(): assert sha256(target)==sha256(source),'续训别名已有不同内容'
        else: atomic_copy(source,target,verify_checkpoint=True)
    print('完整备份恢复完成；保留 teacher 并按原协议 --resume。断线丢失的耗时不能伪造。')
else:
    print('未选择恢复源，没有复制/覆盖文件。')
